[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/visualizacion_datos/01_exploracion/exploracion_hallazgos.ipynb)

# Fase 1: Exploración y Hallazgos
## Detección de Deslizamientos de Tierra con Machine Learning
**Asignatura:** Visualización de Datos · 2026  
**Dataset:** Landslide4Sense — Sentinel-1/2 · ALOS DEM · 14 canales · 3799 muestras  
**Objetivo:** Descubrir qué modelos y qué señales del terreno son más discriminativas para detectar deslizamientos.

---

In [ ]:
# ── Setup: detectar entorno (Colab vs. local) y configurar rutas ──────────────
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if not os.path.exists('Landslides_-Applied-ML-Course'):
        os.system('git clone https://github.com/apmontesp/Landslides_-Applied-ML-Course.git')
    DATA_DIR = 'Landslides_-Applied-ML-Course/visualizacion_datos/data'
    FIG_DIR  = 'Landslides_-Applied-ML-Course/visualizacion_datos/data/figures'
else:
    DATA_DIR = '../data'
    FIG_DIR  = '../data/figures'

os.makedirs(FIG_DIR, exist_ok=True)
print(f'Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'DATA_DIR: {DATA_DIR}')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('Librerias cargadas OK')

In [ ]:
# ── Cargar datos de resultados ────────────────────────────────────────────────
df_models   = pd.read_csv(f'{DATA_DIR}/comparison_table.csv')
df_channels = pd.read_csv(f'{DATA_DIR}/channel_stats_by_class.csv')

with open(f'{DATA_DIR}/final_summary.json') as f:
    summary = json.load(f)

print('Datos cargados:')
print(f'  - Modelos evaluados: {len(df_models)}')
print(f'  - Canales analizados: {len(df_channels)}')
print(f'  - Mejor modelo: {summary["best_model"]} (F1={summary["best_f1"]:.4f})')
print()
df_models

## 1. Pregunta de Negocio

> **¿Qué modelo de Machine Learning detecta mejor los deslizamientos de tierra en imágenes satelitales multiespectrales, y qué características físicas del terreno son más determinantes para la predicción?**

**Contexto:** Los deslizamientos de tierra causan miles de muertes y miles de millones de dólares en pérdidas anuales. Detectarlos automáticamente desde imágenes satelitales permitiría alertas tempranas y mapeo rápido de riesgo. El dataset Landslide4Sense contiene imágenes de 14 bandas (ópticas, SAR, topográficas) etiquetadas como landslide (positivo) o no-landslide (negativo).

**Modelos evaluados:**
- **Clásicos:** Logistic Regression, SVM (RBF), Random Forest
- **Deep Learning:** ResNet-50, EfficientNet-B4, U-Net ResNet-34

**Métricas clave:** F1 Score (balance precisión-recall), AUC-ROC, Recall (prioridad en alertas tempranas)

---
## 2. Exploración 1: Que modelo tiene mayor F1 Score?

**Hipótesis inicial:** Esperamos que los modelos de Deep Learning superen a los clásicos, dado su mayor capacidad de representación.

Comenzamos con una visualización exploratoria directa: barras simples de F1 medio por modelo.

In [ ]:
# ── Exploración 1: Barras de F1 por modelo (estilo exploratorio raw) ──────────
fig, ax = plt.subplots(figsize=(10, 5))

df_sorted = df_models.sort_values('F1 medio', ascending=False)
colors = ['#2ca02c' if t == 'Clásico' else '#9467bd' for t in df_sorted['Tipo']]

bars = ax.bar(df_sorted['Modelo'], df_sorted['F1 medio'],
              color=colors, alpha=0.8, edgecolor='white', linewidth=0.5)

ax.errorbar(df_sorted['Modelo'], df_sorted['F1 medio'],
            yerr=df_sorted['Std'].fillna(0),
            fmt='none', color='#333333', capsize=5, linewidth=1.5)

for bar, val in zip(bars, df_sorted['F1 medio']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, color='black')

ax.set_ylim(0, 1.0)
ax.set_xlabel('Modelo')
ax.set_ylabel('F1 Score Medio (5-fold CV)')
ax.set_title('F1 Score por Modelo — Deteccion de Deslizamientos (exploracion inicial)')
ax.tick_params(axis='x', rotation=20)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ca02c', label='Clasico'),
                   Patch(facecolor='#9467bd', label='Deep Learning')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_1_f1_barras.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: exploracion_1_f1_barras.png')

### Analisis Exploracion 1

**Observacion inesperada:** El modelo **Random Forest (F1=0.837)** supera a todos los modelos de Deep Learning, incluyendo ResNet-50 (F1=0.784) y EfficientNet-B4 (F1=0.755). U-Net ResNet-34 — disenada especificamente para segmentacion semantica — obtiene el peor resultado (F1=0.444).

**Por que?** Exploramos mas en la siguiente visualizacion...

In [ ]:
# ── Analisis complementario: Precision vs Recall (scatter exploratorio) ───────
df_completo = df_models.dropna(subset=['Precisión', 'Recall']).copy()

fig, ax = plt.subplots(figsize=(8, 6))

scatter_colors = {'Clásico': '#2ca02c', 'Deep Learning': '#9467bd'}
for tipo, grupo in df_completo.groupby('Tipo'):
    ax.scatter(grupo['Precisión'], grupo['Recall'],
               c=scatter_colors[tipo], s=grupo['F1 medio']*300,
               label=tipo, alpha=0.85, edgecolors='white', linewidth=1)
    for _, row in grupo.iterrows():
        ax.annotate(row['Modelo'], (row['Precisión'], row['Recall']),
                    textcoords='offset points', xytext=(8, 5), fontsize=9)

for f1 in [0.75, 0.80, 0.85]:
    prec = np.linspace(0.01, 1.0, 300)
    rec = f1 * prec / (2 * prec - f1 + 1e-9)
    mask = (rec > 0) & (rec <= 1) & (prec <= 1)
    ax.plot(prec[mask], rec[mask], '--', color='gray', alpha=0.5, linewidth=1)
    idx = len(prec[mask])//2
    ax.text(prec[mask][idx]+0.01, rec[mask][idx], f'F1={f1}', fontsize=8, color='gray', alpha=0.7)

ax.set_xlabel('Precision')
ax.set_ylabel('Recall')
ax.set_title('Trade-off Precision vs Recall — Modelos Clasicos\n(tamano burbuja proporcional a F1 Score)')
ax.legend()
ax.set_xlim(0.65, 0.90)
ax.set_ylim(0.70, 1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_prec_recall.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Exploración 2: Que canales espectrales discriminan mejor los deslizamientos?

**Pregunta:** Existe diferencia estadistica entre la senal de pixeles de deslizamiento (positivos) vs. no-deslizamiento (negativos) en cada canal?

Calculamos el **delta** (diferencia de medias) como proxy de poder discriminativo.

In [ ]:
# ── Exploración 2: Delta de canales ───────────────────────────────────────────
df_ch = df_channels.copy()
df_ch['|Delta|'] = df_ch['Delta'].abs()
df_ch = df_ch.sort_values('|Delta|', ascending=True)

def sensor_cat(name):
    if 'SAR' in name or 'VV' in name or 'VH' in name: return 'SAR'
    elif 'DEM' in name or 'Slope' in name: return 'Topografia'
    elif 'RedEdge' in name: return 'RedEdge'
    else: return 'Optico'

df_ch['Sensor'] = df_ch['Nombre'].apply(sensor_cat)
palette = {'SAR': '#F59E0B', 'Topografia': '#EF4444', 'RedEdge': '#D62728', 'Optico': '#3B82F6'}

fig, ax = plt.subplots(figsize=(10, 7))
colors_bars = [palette[s] for s in df_ch['Sensor']]
bars = ax.barh(df_ch['Nombre'], df_ch['|Delta|'], color=colors_bars, alpha=0.85, edgecolor='white')

for bar, val in zip(bars, df_ch['|Delta|']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('|Delta media| = |Media_Landslide - Media_NoLandslide|')
ax.set_title('Poder Discriminativo por Canal Espectral\n(exploracion — mayor Delta = mas informativo)')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in palette.items()]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_2_canales_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nTop 5 canales mas discriminativos:')
print(df_ch.sort_values('|Delta|', ascending=False)[['Nombre', 'Sensor', '|Delta|', 'Delta']].head(5).to_string(index=False))

### Analisis Exploracion 2

**Patron claro:** Los canales **RedEdge3 (B7, |Delta|=0.807)** y **RedEdge2 (B6, |Delta|=0.563)** sobresalen notablemente sobre el resto. Estos canales de borde rojo de Sentinel-2 son altamente sensibles a la vegetacion y a suelos expuestos — exactamente lo que caracteriza una zona de deslizamiento fresco.

Los canales opticos convencionales (Azul, Verde, Rojo, NIR) muestran poca diferencia entre clases, lo que explica por que los modelos que priorizan esos canales tienen menor rendimiento.

In [ ]:
# ── Exploración 3: Comparacion media por clase (top 6 canales) ────────────────
df_top6 = df_channels.sort_values('Delta', key=abs, ascending=False).head(6)
x = np.arange(len(df_top6))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, df_top6['Media_Pos'], width, label='Landslide (Positivo)',
       color='#D62728', alpha=0.8, yerr=df_top6['Std_Pos'], capsize=4)
ax.bar(x + width/2, df_top6['Media_Neg'], width, label='No-Landslide (Negativo)',
       color='#1F77B4', alpha=0.8, yerr=df_top6['Std_Neg'], capsize=4)

ax.set_xticks(x)
ax.set_xticklabels(df_top6['Nombre'], rotation=15, ha='right')
ax.set_ylabel('Media de intensidad normalizada')
ax.set_title('Media por Clase — Top 6 Canales Mas Discriminativos')
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_3_medias_clase.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Exploración 4: Ranking F1 y Recall por modelo ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df_sorted = df_models.sort_values('F1 medio', ascending=False).reset_index(drop=True)
colors_list = ['#D62728' if i == 0 else ('#2ca02c' if t == 'Clásico' else '#9467bd')
               for i, t in enumerate(df_sorted['Tipo'])]

axes[0].barh(df_sorted['Modelo'][::-1], df_sorted['F1 medio'][::-1],
             color=colors_list[::-1], alpha=0.85)
axes[0].errorbar(df_sorted['F1 medio'][::-1], df_sorted['Modelo'][::-1],
                 xerr=df_sorted['Std'].fillna(0)[::-1], fmt='none', color='black', capsize=4)
axes[0].axvline(x=0.8, color='gray', linestyle='--', alpha=0.7, label='F1 = 0.80')
axes[0].set_xlabel('F1 Score Medio')
axes[0].set_title('Ranking de Modelos (media +/- std)')
axes[0].legend(fontsize=9)

df_rec = df_models.dropna(subset=['Recall']).sort_values('Recall', ascending=True)
colors_rec = ['#D62728' if m == 'Random Forest' else '#7F7F7F' for m in df_rec['Modelo']]
axes[1].barh(df_rec['Modelo'], df_rec['Recall'], color=colors_rec, alpha=0.85)
axes[1].axvline(x=0.9, color='#D62728', linestyle='--', alpha=0.7, label='Recall = 0.90')
axes[1].set_xlabel('Recall')
axes[1].set_title('Recall por Modelo\n(critico para alertas tempranas)')
axes[1].set_xlim(0.7, 1.0)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/exploracion_4_recall.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Exploración 5: Variabilidad entre folds — consistencia del modelo

**Pregunta:** Las metricas promedio ocultan variabilidad. Un modelo con F1=0.83 pero std=0.03 puede ser poco fiable operativamente. ¿Son los modelos estables entre particiones del dataset?

Cargamos los resultados por fold de cada modelo para comparar su distribucion real.

In [ ]:
# ── Exploración 5: Cargar resultados por fold de cada modelo ──────────────────
import json as _json

# Ruta a results/ relativa al notebook
if IN_COLAB:
    RESULTS_DIR = 'Landslides_-Applied-ML-Course/results'
else:
    RESULTS_DIR = '../../results'

def load_folds_classical(path):
    with open(path) as f:
        d = _json.load(f)
    return [fold['best_f1'] for fold in d['folds']]

def load_folds_dl(path, key='f1_pixel_thr05'):
    with open(path) as f:
        d = _json.load(f)
    return [fold[key] for fold in d['folds']]

fold_data = {}
try:
    fold_data['Logistic Reg.']  = load_folds_classical(f'{RESULTS_DIR}/classical_baselines/logistic_regression/kfold_summary_v2.json')
    fold_data['SVM (RBF)']      = load_folds_classical(f'{RESULTS_DIR}/classical_baselines/svm/kfold_summary_v2.json')
    fold_data['Random Forest']  = load_folds_classical(f'{RESULTS_DIR}/random_forest/kfold_summary_v2.json')
    fold_data['ResNet-50']      = load_folds_dl(f'{RESULTS_DIR}/comparable_literature/resnet50_5fold/kfold5_summary.json', key='f1_thr05')
    fold_data['U-Net ResNet34'] = load_folds_dl(f'{RESULTS_DIR}/comparable_literatura/unet_5fold/kfold5_summary.json', key='f1_pixel_thr05')
    # EfficientNet-B4: sin datos por fold, se omite del analisis de variabilidad
    print('Folds cargados OK:')
    for k, v in fold_data.items():
        print(f'  {k:<20} folds={len(v)}  mean={np.mean(v):.4f}  std={np.std(v):.4f}')
except FileNotFoundError as e:
    print(f'Advertencia ruta no encontrada: {e}')
    fold_data = {}

In [ ]:
# ── Exploración 5a: Boxplot de F1 por modelo (distribucion real 5 folds) ──────
if fold_data:
    order = sorted(fold_data.keys(), key=lambda k: np.median(fold_data[k]), reverse=True)
    data_ordered = [fold_data[k] for k in order]

    VERDE   = '#2ca02c'
    PURPURA = '#9467bd'
    dl_models = {'ResNet-50', 'U-Net ResNet34'}
    box_colors = [PURPURA if m in dl_models else VERDE for m in order]

    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot(data_ordered, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5))

    for patch, color in zip(bp['boxes'], box_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    for i, vals in enumerate(data_ordered, 1):
        ax.scatter([i]*len(vals), vals, color='black', s=30, zorder=5, alpha=0.7)

    ax.set_xticks(range(1, len(order)+1))
    ax.set_xticklabels(order, rotation=15, ha='right')
    ax.set_ylabel('F1 Score (por fold)')
    ax.set_title('Distribucion de F1 Score por Modelo — 5-fold CV\n(cada punto = un fold, caja = rango intercuartil)')
    ax.set_ylim(0.35, 1.0)
    ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5)
    ax.text(len(order) + 0.1, 0.803, 'F1 = 0.80', fontsize=9, color='gray', va='bottom')

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=VERDE, label='Clasico'),
                       Patch(facecolor=PURPURA, label='Deep Learning')]
    ax.legend(handles=legend_elements, loc='lower right')

    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/exploracion_5a_boxplot_folds.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada: exploracion_5a_boxplot_folds.png')

### Analisis Exploracion 5a — Boxplot

**Patron de estabilidad:** Random Forest muestra la caja mas estrecha — sus 5 folds oscilan apenas 0.012 puntos F1 (0.824–0.848). En cambio, SVM (RBF) presenta la mayor dispersion (std=0.030), con el fold 1 cayendo hasta F1=0.753, por debajo del umbral operativo de 0.80.

**U-Net ResNet-34** presenta una distribucion compacta pero en un rango bajo (0.68–0.71), lo que indica que su bajo rendimiento es sistematico y no un artefacto de una mala particion.

In [ ]:
# ── Exploración 5b: Heatmap fold x modelo ─────────────────────────────────────
if fold_data:
    df_folds = pd.DataFrame(fold_data, index=[f'Fold {i}' for i in range(1, 6)])

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.heatmap(df_folds.T, annot=True, fmt='.3f', cmap='RdYlGn',
                vmin=0.40, vmax=0.90, linewidths=0.5, linecolor='white',
                ax=ax, cbar_kws={'label': 'F1 Score'})

    ax.set_title('Heatmap F1 Score — Fold x Modelo\n(verde = mejor rendimiento, rojo = peor)')
    ax.set_xlabel('Particion (fold)')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/exploracion_5b_heatmap_folds.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figura guardada: exploracion_5b_heatmap_folds.png')

    fold_means = df_folds.mean(axis=1)
    worst_fold = fold_means.idxmin()
    print(f'\nFold con menor F1 promedio entre modelos: {worst_fold} (media={fold_means.min():.4f})')
    print('Medias por fold:')
    print(fold_means.round(4).to_string())

### Analisis Exploracion 5b — Heatmap

**Pregunta clave:** Si hay un fold sistematicamente mas dificil para todos los modelos, eso sugiere una particion de datos geográficamente sesgada — ese fold contiene imagenes con caracteristicas espectrales distintas al resto.

La columna con valores mas bajos (mas rojo) identifica la particion mas exigente del dataset. Este analisis complementa la evaluacion LORO (Leave-One-Region-Out) del dashboard interactivo, donde el sesgo geografico se hace aun mas evidente.

---
## 5. El Hallazgo

Despues del analisis exploratorio, se articulan dos hallazgos principales:

---

### Hallazgo 1: Random Forest supera a las redes neuronales profundas

**Anomalia encontrada:** Contra la intuicion dominante en computer vision, el modelo **Random Forest (F1=0.837)** supera a todos los modelos de Deep Learning. Esto se explica porque:
- El dataset Landslide4Sense (3799 muestras) es **relativamente pequeno** para entrenar redes profundas desde cero
- Las features engineered (14 canales ya pre-procesados) dan ventaja a los metodos de ensamble sobre las CNN que deben aprender representaciones
- U-Net (F1=0.444) sufre especialmente por su alta capacidad parametrica sin suficientes datos de entrenamiento

**Implicacion de negocio:** Para sistemas de deteccion con datasets limitados y features bien definidas, los modelos clasicos de ML son la primera eleccion, no las arquitecturas profundas.

---

### Hallazgo 2: Los canales RedEdge (B6, B7) son la senal mas discriminativa

**Correlacion descubierta:** Las bandas espectrales de **borde rojo** (RedEdge) de Sentinel-2 muestran una diferencia de medias (Delta) hasta **10 veces mayor** que los canales opticos convencionales. Esta senal corresponde fisicamente a la respuesta espectral del suelo desnudo expuesto durante un deslizamiento.

**Implicacion de negocio:** Priorizar imagenes con bandas RedEdge disponibles (Sentinel-2 Nivel 2A) maximizara la precision de deteccion. El **DEM de elevacion (Delta=0.195)** y **SAR-VH (Delta=0.188)** complementan la senal optica.

---

**Estos hallazgos guiaran el diseno del Dashboard aclaratorio (Fase 2).**

In [ ]:
# ── Resumen cuantitativo del hallazgo ─────────────────────────────────────────
print('='*60)
print('RESUMEN DE HALLAZGOS — EXPLORACION')
print('='*60)
print()
print('Hallazgo 1: Ranking de modelos por F1 Score')
for _, row in df_models.sort_values('F1 medio', ascending=False).iterrows():
    marker = '>>' if row['Modelo'] == 'Random Forest' else '  '
    print(f'  {marker} {row["Modelo"]:<25} F1={row["F1 medio"]:.4f}  [{row["Tipo"]}]')

print()
print('Hallazgo 2: Top 5 canales mas discriminativos (|Delta media|)')
df_channels_sorted = df_channels.copy()
df_channels_sorted['|Delta|'] = df_channels_sorted['Delta'].abs()
for _, row in df_channels_sorted.sort_values('|Delta|', ascending=False).head(5).iterrows():
    print(f'  Canal {int(row["Canal"]):2d} — {row["Nombre"]:<25} |Delta|={row["|Delta|"]:.4f}')

print()
if fold_data:
    print('Hallazgo 3: Estabilidad entre folds (std F1)')
    for modelo, folds in sorted(fold_data.items(), key=lambda x: np.std(x[1])):
        marker = '>>' if modelo == 'Random Forest' else '  '
        print(f'  {marker} {modelo:<20} mean={np.mean(folds):.4f}  std={np.std(folds):.4f}  [{"ESTABLE" if np.std(folds) < 0.015 else "VARIABLE"}'  + ']')

print()
print('-> Estos hallazgos seran el centro del Dashboard aclaratorio.')
print('='*60)
